# Lesson 01 - Introdução aos Agentes de IA

Bem-vindo à primeira aula do curso **Agentes de IA para Iniciantes**!

Um **agente de IA** é um programa que utiliza um grande modelo de linguagem (LLM) como seu motor de raciocínio e pode tomar *ações* no mundo real — chamar APIs, consultar bases de dados ou executar código — para alcançar um objetivo em nome de um utilizador.

Neste notebook você irá construir o seu primeiro agente: um **Agente de Viagens** que recomenda destinos de férias. Ao longo do caminho, irá aprender a:

1. Ligar-se ao Serviço Azure AI Foundry Agent usando o **Microsoft Agent Framework**.
2. Dar ao agente uma **ferramenta** — uma função Python simples que ele pode chamar.
3. Executar o agente e inspecionar a sua resposta.
4. Transmitir a resposta do agente token a token.


## Configuração

Antes de executar este notebook, certifique-se de que tem:

1. **Um projeto Azure AI Foundry** com um modelo de chat implementado (por exemplo, `gpt-4o-mini`).
2. **Iniciado sessão com a Azure CLI** — execute `az login` no seu terminal.
3. **Definido as variáveis de ambiente necessárias:**
   - `AZURE_AI_PROJECT_ENDPOINT` — o endpoint do seu projeto Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — o nome do seu modelo implementado.

A célula abaixo instala os pacotes Python de que precisa.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Criar o Seu Primeiro Agente

Um agente precisa de duas coisas:

- **Instruções** que lhe digam *quem é* e *como deve comportar-se* (um prompt do sistema).
- **Ferramentas** — funções Python decoradas com `@tool` que o agente pode chamar para obter informação ou executar ações.

Abaixo definimos uma ferramenta simples que devolve uma lista de destinos de férias populares. O agente usará esta ferramenta quando um utilizador pedir recomendações de viagem.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Respostas por Streaming

Para uma experiência mais interativa, pode **transmitir** a resposta do agente em tempo real. Em vez de esperar pela resposta completa, o agente entrega fragmentos de texto conforme são gerados. Isto é especialmente útil em interfaces de chat onde se pretende mostrar a saída em tempo real.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Resumo

Nesta lição aprendeu a:

- **Criar um provider** que se liga ao Azure AI Foundry Agent Service através de `AzureAIProjectAgentProvider`.
- **Definir uma ferramenta** usando o decorador `@tool` para que o agente possa chamar as suas funções em Python.
- **Executar o agente** com uma mensagem do utilizador e imprimir a sua resposta.
- **Transmitir respostas** para saída em tempo real.

Na próxima lição iremos explorar frameworks agentic mais a fundo e aprender como dar aos agentes ferramentas mais poderosas e capacidades de raciocínio em múltiplas fases.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
